# F5 Best: Full-image vs Tiled Inference trên VisDrone

Notebook **không train lại**. Notebook tải đúng F5 `best.pth` epoch 5, chạy baseline full-image và tiled inference 640×640 overlap 20% trên cùng 548 ảnh validation, đánh giá bằng cùng VisDrone DET evaluator và tự kết luận có đáng làm F6 hay không.

Trước khi Run All: bật **GPU** và **Internet**, sau đó Add Input gồm (1) Kaggle Output có `f5/best.pth`, (2) raw `VisDrone2019-DET-val` có `images/` và `annotations/`.

## 1. Cấu hình

Nếu auto-discovery chọn sai checkpoint hoặc raw validation root, điền hai biến override bằng đường dẫn trong `/kaggle/input`.

In [ ]:
from pathlib import Path

CHECKPOINT_OVERRIDE = None  # ví dụ: /kaggle/input/<f5-output>/faster_rcnn_runs/f5/best.pth
RAW_VAL_ROOT_OVERRIDE = None  # thư mục có images/ và annotations/

EXPECTED_EXPERIMENT = "F5"
EXPECTED_BEST_EPOCH = 5
RUN_FULL_BASELINE = True
RUN_TILED = True

TILE_SIZE = 640
TILE_OVERLAP = 0.20
TILE_BATCH_SIZE = 1  # giữ 1 trên Tesla T4 để tránh OOM
MERGE_NMS_IOU = 0.50
PRE_NMS_TOPK = 10000
WORKERS = 2

OUTPUT_ROOT = Path("/kaggle/working/f5_best_tiled_experiment")
REPO_URL = "https://github.com/Nhattk19/Object_dectection.git"
REPO_BRANCH = "cnn-faster-rcnn-pipeline"

## 2. Lấy code mới nhất và kiểm tra GPU

In [ ]:
import subprocess
import sys

PROJECT_ROOT = Path("/kaggle/working/Object_dectection_tiled")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
elif not (PROJECT_ROOT / "scripts/evaluate_visdrone_checkpoint.py").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)])

help_text = subprocess.check_output(
    [sys.executable, str(PROJECT_ROOT / "scripts/evaluate_visdrone_checkpoint.py"), "--help"],
    text=True,
)
assert "--tile-size" in help_text, "Repository chưa có tiled evaluator; hãy pull/push code mới rồi Restart Session"

import torch
assert torch.cuda.is_available(), "Vào Settings bật GPU Accelerator"
print({"project": str(PROJECT_ROOT), "gpu": torch.cuda.get_device_name(0)})

## 3. Tìm và xác nhận F5 best epoch 5 + raw validation

In [ ]:
import gc

def valid_raw_val(path):
    path = Path(path)
    return (
        (path / "images").is_dir()
        and (path / "annotations").is_dir()
        and len(list((path / "images").glob("*.jpg"))) >= 500
        and len(list((path / "annotations").glob("*.txt"))) >= 500
    )

if RAW_VAL_ROOT_OVERRIDE:
    RAW_ROOT = Path(RAW_VAL_ROOT_OVERRIDE)
else:
    RAW_ROOT = next(
        (p.parent for p in Path("/kaggle/input").glob("**/annotations") if valid_raw_val(p.parent)),
        None,
    )
if RAW_ROOT is None or not valid_raw_val(RAW_ROOT):
    raise FileNotFoundError("Không tìm thấy raw VisDrone val có images/*.jpg và annotations/*.txt")

if CHECKPOINT_OVERRIDE:
    checkpoint_candidates = [Path(CHECKPOINT_OVERRIDE)]
else:
    checkpoint_candidates = sorted(
        p for p in Path("/kaggle/input").glob("**/best.pth")
        if p.parent.name.lower() == "f5" and "smoke" not in str(p).lower()
    )
if not checkpoint_candidates:
    raise FileNotFoundError("Không tìm thấy f5/best.pth; hãy Add Output F5 hoặc đặt CHECKPOINT_OVERRIDE")

CHECKPOINT = None
CHECKPOINT_METADATA = None
for candidate in checkpoint_candidates:
    payload = torch.load(candidate, map_location="cpu", weights_only=False)
    config = payload.get("config", {})
    if config.get("experiment") == EXPECTED_EXPERIMENT and int(payload.get("epoch", -1)) == EXPECTED_BEST_EPOCH:
        CHECKPOINT = candidate
        CHECKPOINT_METADATA = {
            "experiment": config.get("experiment"),
            "epoch": int(payload["epoch"]),
            "selection_metric": config.get("selection_metric"),
            "metrics": payload.get("metrics", {}),
            "min_size": config.get("min_size"),
            "max_size": config.get("max_size"),
        }
        del payload
        break
    del payload
    gc.collect()

if CHECKPOINT is None:
    raise RuntimeError(f"Không có checkpoint F5 best epoch 5 trong: {checkpoint_candidates}")
assert CHECKPOINT_METADATA["selection_metric"] == "visdrone_ap"
assert CHECKPOINT_METADATA["min_size"] == 896 and CHECKPOINT_METADATA["max_size"] == 1493
print({"checkpoint": str(CHECKPOINT), "raw_val": str(RAW_ROOT), **CHECKPOINT_METADATA})

## 4. Baseline full-image bằng chính F5 best

Bước này tái tạo prediction chính thức của epoch 5. Có thể đặt `RUN_FULL_BASELINE=False` ở cell cấu hình nếu đã có output baseline từ đúng checkpoint và đúng evaluator.

In [ ]:
import time

FULL_RUN_NAME = "f5_best_full"
if RUN_FULL_BASELINE:
    full_command = [
        sys.executable, "-u", str(PROJECT_ROOT / "scripts/evaluate_visdrone_checkpoint.py"),
        "--checkpoint", str(CHECKPOINT),
        "--raw-val-root", str(RAW_ROOT),
        "--output-dir", str(OUTPUT_ROOT),
        "--run-name", FULL_RUN_NAME,
        "--workers", str(WORKERS),
        "--score-threshold", "0.001",
    ]
    started = time.time()
    print("Running full-image baseline:", " ".join(full_command))
    subprocess.check_call(full_command, cwd=PROJECT_ROOT)
    print(f"Full-image finished in {(time.time() - started) / 60:.1f} minutes")
else:
    print("Skipped full-image baseline")

## 5. Tiled inference 640×640, overlap 20%

Mỗi tile vẫn đi qua transform 896/1493 lưu trong F5 checkpoint. Bounding box được đưa về tọa độ toàn ảnh, sau đó gộp class-wise bằng NMS IoU 0,50 và giữ tối đa 500 detection/ảnh. Bước này chậm hơn full-image nhiều lần.

In [ ]:
TILED_RUN_NAME = f"f5_best_tiled_{TILE_SIZE}_o{int(TILE_OVERLAP * 100)}"
if RUN_TILED:
    tiled_command = [
        sys.executable, "-u", str(PROJECT_ROOT / "scripts/evaluate_visdrone_checkpoint.py"),
        "--checkpoint", str(CHECKPOINT),
        "--raw-val-root", str(RAW_ROOT),
        "--output-dir", str(OUTPUT_ROOT),
        "--run-name", TILED_RUN_NAME,
        "--workers", str(WORKERS),
        "--score-threshold", "0.001",
        "--tile-size", str(TILE_SIZE),
        "--tile-overlap", str(TILE_OVERLAP),
        "--tile-batch-size", str(TILE_BATCH_SIZE),
        "--merge-nms-iou", str(MERGE_NMS_IOU),
        "--pre-nms-topk", str(PRE_NMS_TOPK),
    ]
    started = time.time()
    print("Running tiled inference:", " ".join(tiled_command))
    subprocess.check_call(tiled_command, cwd=PROJECT_ROOT)
    print(f"Tiled inference finished in {(time.time() - started) / 60:.1f} minutes")
else:
    print("Skipped tiled inference")

## 6. So sánh và quyết định F6

Tiêu chí đề xuất: tiled inference phải tăng ít nhất 1,0 điểm AP hoặc 1,5 điểm AR@500 mới đáng đầu tư tiled training cho F6.

In [ ]:
import json
import pandas as pd
from IPython.display import display

full_path = OUTPUT_ROOT / FULL_RUN_NAME / "visdrone_metrics.json"
tiled_path = OUTPUT_ROOT / TILED_RUN_NAME / "visdrone_metrics.json"
if not full_path.is_file() or not tiled_path.is_file():
    raise FileNotFoundError("Cần chạy cả baseline và tiled trước khi so sánh")

full_report = json.loads(full_path.read_text())
tiled_report = json.loads(tiled_path.read_text())
metric_keys = [
    "visdrone_ap", "visdrone_ap50", "visdrone_ap75",
    "visdrone_ar1", "visdrone_ar10", "visdrone_ar100", "visdrone_ar500",
]
comparison = pd.DataFrame({
    "metric": metric_keys,
    "full_image_%": [100 * full_report[key] for key in metric_keys],
    "tiled_%": [100 * tiled_report[key] for key in metric_keys],
})
comparison["delta_pp"] = comparison["tiled_%"] - comparison["full_image_%"]
display(comparison.round(4))

delta_ap = tiled_report["visdrone_ap"] - full_report["visdrone_ap"]
delta_ar500 = tiled_report["visdrone_ar500"] - full_report["visdrone_ar500"]
recommend_f6 = delta_ap >= 0.01 or delta_ar500 >= 0.015
decision = {
    "f5_checkpoint_epoch": full_report["checkpoint_epoch"],
    "delta_ap_pp": 100 * delta_ap,
    "delta_ar500_pp": 100 * delta_ar500,
    "recommend_f6_tiled_training": recommend_f6,
    "conclusion": (
        "Nên chuẩn bị F6 = F5 + tiled training/inference"
        if recommend_f6
        else "Không nên train F6 tiling; giữ F5 best full-image"
    ),
}
print(json.dumps(decision, indent=2, ensure_ascii=False))
(OUTPUT_ROOT / "f5_tiled_decision.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False))
comparison.to_csv(OUTPUT_ROOT / "f5_full_vs_tiled.csv", index=False)

## 7. Đóng gói Kaggle Output

Sau khi cell hoàn tất, chọn **Save Version**. Không cần tải từng TXT riêng lẻ.

In [ ]:
import shutil

archive = shutil.make_archive(
    "/kaggle/working/f5_best_tiled_experiment",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print({
    "output_root": str(OUTPUT_ROOT),
    "comparison_csv": str(OUTPUT_ROOT / "f5_full_vs_tiled.csv"),
    "decision_json": str(OUTPUT_ROOT / "f5_tiled_decision.json"),
    "archive": archive,
})